In [0]:
%sql

CREATE OR REPLACE TABLE traffic_catalog.gold.gold_traffic_summary
USING DELTA
AS

SELECT
    DATE(event_timestamp) AS measurement_date,

    COUNT(*) AS total_measurements,

    COUNT(DISTINCT road_id) AS roads_monitored,

    ROUND(AVG(current_speed), 2) AS avg_speed,

    ROUND(AVG(free_flow_speed), 2) AS avg_free_flow_speed,

    ROUND(
        (
            1 - AVG(current_speed) /
            NULLIF(AVG(free_flow_speed), 0)
        ) * 100,
        2
    ) AS avg_speed_reduction,

    SUM(
        CASE
            WHEN current_speed / NULLIF(free_flow_speed, 0) < 0.70
                THEN 1
            ELSE 0
        END
    ) AS congested_measurements,

    SUM(
        CASE
            WHEN road_closure = true THEN 1
            ELSE 0
        END
    ) AS closure_events,

    ROUND(AVG(confidence), 3) AS avg_confidence

FROM traffic_catalog.silver.silver_traffic

GROUP BY
    DATE(event_timestamp)

ORDER BY
    measurement_date;